In [1]:
# %%
# =============================================================================
# 12_rag_reranking.ipynb
# Financial AI Governance — RAG + Cross-Encoder Re-ranking (Option D)
# Kernel : Python (llm_env)
# Input  : data/processed/dataset_final.json
#          vectordb/ (Chroma persistent stores from 02_rag_pipeline.ipynb)
# Output : results/responses/responses_rag_rerank.json
#          results/tables/table_rerank_summary.csv
# Note   : ChromaDB retrieves top-k=5 (wider net), then cross-encoder
#          re-ranks and selects top-3 for inference.
#          Inference prompt identical to 03_llm_inference.ipynb (RAG).
#          cross-encoder/ms-marco-MiniLM-L-6-v2 runs locally — no API cost.
#          Install: pip install sentence-transformers
# =============================================================================

# %%
# =============================================================================
# Cell 1. Libraries and Environment Setup
# =============================================================================
import os
import json
import time
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv
from tqdm import tqdm

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from sentence_transformers import CrossEncoder

# Directory paths — identical to 03_llm_inference.ipynb
DATA_DIR     = '../data/processed'
VDB_DIR      = '../vectordb'
RESPONSE_DIR = '../results/responses'
TABLE_DIR    = '../results/tables'

for d in [RESPONSE_DIR, TABLE_DIR]:
    os.makedirs(d, exist_ok=True)

# API setup
load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
LLM_MODEL      = os.getenv('LLM_MODEL', 'gpt-4o-mini')
EMBED_MODEL    = 'text-embedding-3-small'

if not OPENAI_API_KEY:
    raise ValueError("[ERROR] OPENAI_API_KEY not set in .env")

client     = OpenAI(api_key=OPENAI_API_KEY)
embeddings = OpenAIEmbeddings(model=EMBED_MODEL, api_key=OPENAI_API_KEY)

# Re-ranking configuration
RETRIEVE_K  = 5   # initial retrieval — wider net
RERANK_TOP  = 3   # final selection after re-ranking
RERANK_MODEL = 'cross-encoder/ms-marco-MiniLM-L-6-v2'  # local, no API cost

print(f"[INFO] LLM model       : {LLM_MODEL}")
print(f"[INFO] Embedding model : {EMBED_MODEL}")
print(f"[INFO] Rerank model    : {RERANK_MODEL} (local)")
print(f"[INFO] Retrieve top-k  : {RETRIEVE_K}  →  rerank to top-{RERANK_TOP}")


# %%
# =============================================================================
# Cell 2. Load Cross-Encoder Re-ranker
# =============================================================================
print(f"[INFO] Loading cross-encoder: {RERANK_MODEL} ...")
reranker = CrossEncoder(RERANK_MODEL)
print(f"[INFO] Cross-encoder loaded.")

# Quick sanity check
test_pairs = [
    ("What are human oversight requirements for AI?",
     "### Article 14 — Human Oversight  High-risk AI systems shall be designed..."),
    ("What are human oversight requirements for AI?",
     "### Article 99 — Penalties  Non-compliance penalties range from EUR 7.5M..."),
]
test_scores = reranker.predict(test_pairs)
print(f"\n[CHECK] Re-ranker sanity check:")
print(f"  Relevant chunk score   : {test_scores[0]:.4f}")
print(f"  Irrelevant chunk score : {test_scores[1]:.4f}")
print(f"  Gap (higher = better)  : {test_scores[0] - test_scores[1]:.4f}")
assert test_scores[0] > test_scores[1], \
    "[ERROR] Re-ranker not working — relevant chunk scored lower than irrelevant"
print(f"[OK] Re-ranker functioning correctly.")


# %%
# =============================================================================
# Cell 3. Load Dataset and Vector Stores
# =============================================================================
with open(os.path.join(DATA_DIR, 'dataset_final.json'), 'r', encoding='utf-8') as f:
    dataset = json.load(f)
df = pd.DataFrame(dataset)
print(f"[INFO] Dataset loaded: {len(df)} records")

VDB_CONFIG = {
    'NIST_AI_RMF'    : {'persist_dir': os.path.join(VDB_DIR, 'nist'),          'collection': 'nist_ai_rmf'},
    'KR_AI_BASIC_ACT': {'persist_dir': os.path.join(VDB_DIR, 'kr_aibasicact'), 'collection': 'kr_aibasicact'},
    'EU_AI_ACT'      : {'persist_dir': os.path.join(VDB_DIR, 'eu_aiact'),      'collection': 'eu_aiact'},
}

vector_stores = {}
for reg_key, cfg in VDB_CONFIG.items():
    vector_stores[reg_key] = Chroma(
        collection_name    = cfg['collection'],
        embedding_function = embeddings,
        persist_directory  = cfg['persist_dir'],
    )
    count = vector_stores[reg_key]._collection.count()
    print(f"  [LOAD] {reg_key:20s} | {count} chunks")

print("[INFO] All vector stores loaded.")


# %%
# =============================================================================
# Cell 4. Prompt Templates (identical to 03_llm_inference.ipynb)
# =============================================================================
SYSTEM_PROMPT = """You are an expert AI governance advisor specializing in financial institution AI compliance.
Your role is to support an AI Review Committee at a financial institution by providing accurate,
regulation-grounded answers to governance questions.

When answering:
1. Cite specific regulatory provisions (article numbers, section codes) where applicable.
2. Identify the governance axis: G1 (Accuracy), G2 (Safety), G3 (Transparency), or G4 (Compliance).
3. Flag high-risk scenarios and recommend human oversight where appropriate.
4. If uncertain, state limitations clearly rather than fabricating information.
5. Keep answers concise, structured, and actionable for a compliance committee."""


def build_rag_prompt(question: str, context: str) -> str:
    """Identical to 03_llm_inference.ipynb build_rag_prompt()."""
    return f"""Answer the following AI governance question using the regulatory context provided below.

--- REGULATORY CONTEXT ---
{context}
--- END CONTEXT ---

Question: {question}

Provide a structured answer grounded in the regulatory context above.
Cite specific article numbers or section codes from the context where applicable."""


# %%
# =============================================================================
# Cell 5. Re-ranking Retrieval and Inference Functions
# =============================================================================
def retrieve_and_rerank(question: str, regulation: str,
                        retrieve_k: int = RETRIEVE_K,
                        rerank_top: int = RERANK_TOP) -> tuple[str, list]:
    """
    Two-stage retrieval:
      Stage 1 — ChromaDB similarity search: retrieve_k candidates (wider net)
      Stage 2 — Cross-encoder re-ranking: select rerank_top best chunks

    Args:
        question    : Question string
        regulation  : Regulation key (e.g., 'NIST_AI_RMF')
        retrieve_k  : Number of candidates from ChromaDB (default: 5)
        rerank_top  : Number of chunks after re-ranking (default: 3)

    Returns:
        context     : Concatenated top-rerank_top chunks
        rerank_scores: List of (chunk_preview, score) for logging
    """
    if regulation not in vector_stores:
        raise ValueError(f"[ERROR] Unknown regulation: {regulation}")

    # Stage 1: Dense retrieval
    docs = vector_stores[regulation].similarity_search(question, k=retrieve_k)
    chunks = [d.page_content for d in docs]

    # Stage 2: Cross-encoder re-ranking
    pairs  = [(question, chunk) for chunk in chunks]
    scores = reranker.predict(pairs)

    # Sort by score descending, select top-rerank_top
    ranked = sorted(zip(chunks, scores), key=lambda x: x[1], reverse=True)
    top_chunks = [chunk for chunk, _ in ranked[:rerank_top]]
    top_scores = [(chunk[:60].replace('\n', ' '), round(float(score), 4))
                  for chunk, score in ranked[:rerank_top]]

    context = "\n\n---\n\n".join(top_chunks)
    return context, top_scores


def call_llm(system_prompt: str, user_prompt: str,
             model: str = LLM_MODEL,
             temperature: float = 0.0,
             max_tokens: int = 1000) -> dict:
    """Identical to 03_llm_inference.ipynb call_llm()."""
    try:
        res = client.chat.completions.create(
            model       = model,
            temperature = temperature,
            max_tokens  = max_tokens,
            messages    = [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user',   'content': user_prompt},
            ]
        )
        return {
            'response'         : res.choices[0].message.content.strip(),
            'prompt_tokens'    : res.usage.prompt_tokens,
            'completion_tokens': res.usage.completion_tokens,
            'total_tokens'     : res.usage.total_tokens,
        }
    except Exception as e:
        return {
            'response'         : f'[ERROR] {str(e)}',
            'prompt_tokens'    : 0,
            'completion_tokens': 0,
            'total_tokens'     : 0,
        }


# %%
# =============================================================================
# Cell 6. Re-ranking Quality Spot-Check
# =============================================================================
TEST_QUERIES = [
    {'question'  : 'What are the requirements for human oversight of '
                   'high-risk AI credit scoring systems?',
     'regulation': 'EU_AI_ACT'},
    {'question'  : 'What obligations apply to AI business operators '
                   'providing High-Impact AI for credit screening?',
     'regulation': 'KR_AI_BASIC_ACT'},
    {'question'  : 'How does the GOVERN function address legal and '
                   'regulatory requirements for AI systems?',
     'regulation': 'NIST_AI_RMF'},
]

print("[INFO] Re-ranking Quality Spot-Check\n")
print("=" * 70)

for t in TEST_QUERIES:
    # Standard RAG k=3 (no reranking)
    docs_std = vector_stores[t['regulation']].similarity_search(
        t['question'], k=3)
    std_titles = [d.page_content[:60].replace('\n', ' ')
                  for d in docs_std]

    # Re-ranked: retrieve k=5, rerank to top-3
    ctx_rr, scores_rr = retrieve_and_rerank(
        t['question'], t['regulation'])

    print(f"Question   : {t['question'][:70]}...")
    print(f"Regulation : {t['regulation']}")
    print(f"  Standard RAG top-3:")
    for i, title in enumerate(std_titles, 1):
        print(f"    [{i}] {title}...")
    print(f"  Re-ranked top-3 (from k=5 candidates):")
    for i, (title, score) in enumerate(scores_rr, 1):
        print(f"    [{i}] score={score:.4f} | {title}...")
    print("-" * 70)


# %%
# =============================================================================
# Cell 7. Run RAG + Re-ranking Inference
# =============================================================================
print(f"[RUN] RAG + Re-ranking inference")
print(f"      Model: {LLM_MODEL} | Temperature: 0.0 | Max tokens: 1000")
print(f"      Retrieve k={RETRIEVE_K} → rerank to top-{RERANK_TOP}")
print(f"      Total records: {len(df)}\n")

results      = []
total_tokens = 0

for _, row in tqdm(df.iterrows(), total=len(df), desc='RAG+Rerank'):
    # Two-stage retrieval
    context, rerank_scores = retrieve_and_rerank(
        row['question'], row['regulation'],
        retrieve_k=RETRIEVE_K,
        rerank_top=RERANK_TOP,
    )

    # Inference — identical to 03_llm_inference.ipynb
    user_prompt = build_rag_prompt(row['question'], context)
    result      = call_llm(SYSTEM_PROMPT, user_prompt)

    # Field structure identical to 03_llm_inference.ipynb
    results.append({
        'id'               : row['id'],
        'scenario_id'      : row['scenario_id'],
        'regulation'       : row['regulation'],
        'function'         : row['function'],
        'difficulty'       : row['difficulty'],
        'financial_domain' : row['financial_domain'],
        'risk_level'       : row['risk_level'],
        'governance_axis'  : row['governance_axis'],
        'question'         : row['question'],
        'ground_truth'     : row['ground_truth'],
        'legal_basis'      : row['legal_basis'],
        'condition'        : 'rag_rerank',
        'retrieve_k'       : RETRIEVE_K,
        'rerank_top'       : RERANK_TOP,
        'rerank_scores'    : rerank_scores,   # additional field
        'context_used'     : context,
        'response'         : result['response'],
        'prompt_tokens'    : result['prompt_tokens'],
        'completion_tokens': result['completion_tokens'],
        'total_tokens'     : result['total_tokens'],
        'model'            : LLM_MODEL,
        'temperature'      : 0.0,
    })

    total_tokens += result['total_tokens']
    time.sleep(0.3)

    idx = len(results)
    if idx % 50 == 0:
        errors  = sum(1 for r in results if r['response'].startswith('[ERROR]'))
        avg_len = sum(len(r['response']) for r in results) / idx
        print(f"  [Checkpoint {idx:3d}/300] errors: {errors} | "
              f"avg response: {avg_len:.0f} chars | "
              f"tokens so far: {total_tokens:,}")

# Save
out_path = os.path.join(RESPONSE_DIR, 'responses_rag_rerank.json')
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

errors = sum(1 for r in results if r['response'].startswith('[ERROR]'))
print(f"\n[SAVE] responses_rag_rerank.json")
print(f"[INFO] records: {len(results)} | errors: {errors} | "
      f"total tokens: {total_tokens:,} | "
      f"estimated cost: ${total_tokens * 0.00000015:.4f}")


# %%
# =============================================================================
# Cell 8. Re-ranking Score Analysis
# =============================================================================
df_results = pd.DataFrame(results)

# Average top-1 rerank score by regulation
# (higher = cross-encoder more confident the top chunk is relevant)
print("[INFO] Re-ranking Score Analysis\n")

reg_scores = {}
for reg in df_results['regulation'].unique():
    reg_data = df_results[df_results['regulation'] == reg]
    top1_scores = [
        r['rerank_scores'][0][1]
        for r in reg_data.to_dict('records')
        if r.get('rerank_scores')
    ]
    reg_scores[reg] = {
        'mean_top1' : round(sum(top1_scores) / len(top1_scores), 4),
        'min_top1'  : round(min(top1_scores), 4),
        'max_top1'  : round(max(top1_scores), 4),
        'n'         : len(top1_scores),
    }

print(f"  {'Regulation':20s} | {'Mean Top-1':>10} | "
      f"{'Min':>8} | {'Max':>8} | N")
print("-" * 60)
for reg, s in reg_scores.items():
    print(f"  {reg:20s} | {s['mean_top1']:>10.4f} | "
          f"{s['min_top1']:>8.4f} | {s['max_top1']:>8.4f} | {s['n']}")


# %%
# =============================================================================
# Cell 9. Summary Comparison — Standard RAG vs RAG+Rerank
# =============================================================================
rag_k3_path = os.path.join(RESPONSE_DIR, 'responses_rag.json')
with open(rag_k3_path, 'r', encoding='utf-8') as f:
    rag_k3_data = json.load(f)
print(f"[LOAD] responses_rag.json: {len(rag_k3_data)} records")

comparison = {
    'rag_k3'    : rag_k3_data,
    'rag_rerank': results,
}

rows = []
for label, data in comparison.items():
    avg_ctx   = sum(len(r.get('context_used', '')) for r in data) / len(data)
    avg_resp  = sum(len(r.get('response', ''))      for r in data) / len(data)
    avg_tok   = sum(r.get('total_tokens', 0)        for r in data) / len(data)
    total_tok = sum(r.get('total_tokens', 0)        for r in data)
    errors    = sum(1 for r in data if r.get('response','').startswith('[ERROR]'))
    rows.append({
        'Condition'           : label,
        'N'                   : len(data),
        'Retrieve-k / Final-k': '3/3' if label == 'rag_k3' else f'{RETRIEVE_K}/{RERANK_TOP}',
        'Avg Context (chars)' : round(avg_ctx),
        'Avg Response (chars)': round(avg_resp),
        'Avg Total Tokens'    : round(avg_tok),
        'Total Tokens'        : total_tok,
        'Errors'              : errors,
        'Est. Cost ($)'       : round(total_tok * 0.00000015, 4),
    })

df_summary = pd.DataFrame(rows)
print("\n[Table] Standard RAG vs RAG+Rerank Summary")
print(df_summary.to_string(index=False))

out_tbl = os.path.join(TABLE_DIR, 'table_rerank_summary.csv')
df_summary.to_csv(out_tbl, index=False, encoding='utf-8-sig')
print(f"\n[SAVE] table_rerank_summary.csv")

print(f"\n✅ Notebook 12 complete — Next: 13_extended_evaluation.ipynb")
print(f"   All response files ready for evaluation:")
print(f"     responses_baseline.json       (Baseline, n=300)")
print(f"     responses_rag.json            (RAG k=3, n=300)")
print(f"     responses_rag_k1.json         (RAG k=1, n=300)")
print(f"     responses_rag_k5.json         (RAG k=5, n=300)")
print(f"     responses_rag_rewrite.json    (RAG+Rewrite, n=300)")
print(f"     responses_rag_rerank.json     (RAG+Rerank, n=300)")
print(f"     responses_rag_gpt4o.json      (RAG gpt-4o, n=90)")
print(f"     responses_rag_mini_sample.json(RAG gpt-4o-mini, n=90)")

[INFO] LLM model       : gpt-4o-mini
[INFO] Embedding model : text-embedding-3-small
[INFO] Rerank model    : cross-encoder/ms-marco-MiniLM-L-6-v2 (local)
[INFO] Retrieve top-k  : 5  →  rerank to top-3
[INFO] Loading cross-encoder: cross-encoder/ms-marco-MiniLM-L-6-v2 ...


Loading weights: 100%|█████████████████████████████████████████████████████████████| 105/105 [00:00<00:00, 7508.09it/s]


[INFO] Cross-encoder loaded.

[CHECK] Re-ranker sanity check:
  Relevant chunk score   : 3.7535
  Irrelevant chunk score : -11.3461
  Gap (higher = better)  : 15.0996
[OK] Re-ranker functioning correctly.
[INFO] Dataset loaded: 300 records
  [LOAD] NIST_AI_RMF          | 11 chunks
  [LOAD] KR_AI_BASIC_ACT      | 17 chunks
  [LOAD] EU_AI_ACT            | 23 chunks
[INFO] All vector stores loaded.
[INFO] Re-ranking Quality Spot-Check

Question   : What are the requirements for human oversight of high-risk AI credit s...
Regulation : EU_AI_ACT
  Standard RAG top-3:
    [1] ### Article 14 — Human Oversight   **Chapter III, Section 2:...
    [2] ### Article 10 — Data and Data Governance   **Chapter III, S...
    [3] ### Article 6 — Classification Rules for High-Risk AI System...
  Re-ranked top-3 (from k=5 candidates):
    [1] score=6.2185 | ### Article 14 — Human Oversight   **Chapter III, Section 2:...
    [2] score=5.1255 | ### Article 13 — Transparency and Provision of Information t...


RAG+Rerank:  17%|███████████▌                                                         | 50/300 [07:16<38:19,  9.20s/it]

  [Checkpoint  50/300] errors: 0 | avg response: 2840 chars | tokens so far: 105,451


...

  [Checkpoint 200/300] errors: 0 | avg response: 2708 chars | tokens so far: 399,971


RAG+Rerank:  83%|████████████████████████████████████████████████████████▋           | 250/300 [37:37<06:34,  7.88s/it]

  [Checkpoint 250/300] errors: 0 | avg response: 2660 chars | tokens so far: 501,906


RAG+Rerank: 100%|████████████████████████████████████████████████████████████████████| 300/300 [45:51<00:00,  9.17s/it]

  [Checkpoint 300/300] errors: 0 | avg response: 2661 chars | tokens so far: 605,759

[SAVE] responses_rag_rerank.json
[INFO] records: 300 | errors: 0 | total tokens: 605,759 | estimated cost: $0.0909
[INFO] Re-ranking Score Analysis

  Regulation           | Mean Top-1 |      Min |      Max | N
------------------------------------------------------------
  NIST_AI_RMF          |    -1.5701 |  -7.2141 |   3.5621 | 100
  KR_AI_BASIC_ACT      |     1.1749 |  -9.7470 |   7.9351 | 100
  EU_AI_ACT            |     1.5820 |  -6.5983 |   7.8518 | 100
[LOAD] responses_rag.json: 300 records

[Table] Standard RAG vs RAG+Rerank Summary
 Condition   N Retrieve-k / Final-k  Avg Context (chars)  Avg Response (chars)  Avg Total Tokens  Total Tokens  Errors  Est. Cost ($)
    rag_k3 300                  3/3                 6457                  2692              2044        613318       0         0.0920
rag_rerank 300                  5/3                 6350                  2661              2019   